# Assignment 03

**Specifications and requirements** for each assignment include compliance with the [Programmer's Pact](../../programmers-pact.md).

This assignment comprises the following problems:

* Dynamic Array Accessors
* Dynamic Array Magic Methods
* Cards, Properties and Decorators
* Bounded Arrays: Inheritance and Name Mangling

## What to submit

A single Python file called `week03.py`.

**This assignment starts from a template, not a blank file.** Download [`week03.py`](./week03.py) and keep it in a folder of its own. The template already contains three classes and a decorator, partly written for you. Your job has two parts:

1. **Fill in every method whose body is a single line reading** `raise NotImplementedError(...)`. There are 20 of them, spread across the four problems.
2. **Add one line**: in Problem 3 you apply a decorator yourself. Search the file for `TODO` -- it is the only one -- and it tells you exactly what to add and where.

The placeholder line you are replacing looks like this:

```python
raise NotImplementedError("replace this line with your implementation")
```

Replace each of those lines with your own implementation. **Any method still holding a `NotImplementedError` counts as unanswered** and will fail its tests -- the placeholder exists so that an unfinished file still imports and type-checks cleanly, not as something to leave behind.

Do **not** rename the class or its methods, change any method's signature, or modify the methods that are already written. The autograder calls the template's names exactly as given.

**There is exactly one exception, in Problem 3:** you add a single `@announce` line inside the `if __name__ == "__main__":` block at the bottom of the file. It is marked with a `TODO` comment telling you exactly where. That is the only line you add rather than replace.

Your solutions should be based on material covered so far in this course: loops, conditionals, lists, strings, functions, and classes.

Use **sensible type annotations** on parameters and return values. The template's signatures are already annotated -- keep them that way. "Sensible" means a type that actually describes the value (`index: int`, `-> bool`, ...), not a blanket `Any` used to dodge the requirement; your annotations are checked for internal consistency, not just presence.

## Generative AI policy

You may use generative AI tools to get hints or to clarify concepts, provided you **cite** the tool and how you used it (a short comment at the top of `week03.py` is sufficient, e.g. `# Used ChatGPT to clarify how Python's len() and __len__ relate`). You may **not** use generative AI to generate a solution to any of these problems verbatim -- the code you submit must be your own.

### Turn off AI assistance in your editor before you start

**This is required, not a suggestion.** Autocomplete that writes the next line for you is generative AI, even when it does not feel like it. An inline suggestion that finishes `remove` while you are still working out the shift is the same thing as pasting in a generated solution, and it defeats the entire point of the assignment. Turn these features off before you open `week03.py`, and leave them off until you submit.

**The reliable way -- disable the extensions.** In VS Code, open the Extensions view (`Cmd+Shift+X` on macOS, `Ctrl+Shift+X` on Windows/Linux), search for each of the following, and click **Disable**:

- **GitHub Copilot**
- **GitHub Copilot Chat**
- any other AI assistant you have installed (Codeium / Windsurf, Tabnine, Amazon Q, Continue, Cline, and so on)

Then open the Command Palette (`Cmd+Shift+P` / `Ctrl+Shift+P`), run **GitHub Copilot: Sign Out** if it appears, and check that the Copilot icon in the status bar is gone or shows as disabled.

**Also switch off generic inline suggestions.** Open Settings (`Cmd+,` / `Ctrl+,`), search for `inline suggest`, and uncheck **Editor > Inline Suggest: Enabled**. Equivalently, add this to your `settings.json`:

```json
{
    "github.copilot.enable": { "*": false },
    "editor.inlineSuggest.enabled": false
}
```

Setting names move around between VS Code releases, so if you cannot find one of these, disabling the extensions above is what actually matters.

**Using a different editor?** The same rule applies. Cursor and Windsurf are AI editors -- turn off Tab autocomplete and the chat sidebar, or just use plain VS Code for this assignment. In PyCharm, disable **JetBrains AI Assistant** and **Full Line Code Completion**.

**Check that it worked.** Open `week03.py`, put your cursor on a blank line inside one of the unimplemented methods, and wait a few seconds. If greyed-out code appears ahead of your cursor, something is still switched on.

Working without these tools is the point. You are learning to build the data structure, not to accept a suggestion for it, and the exam will not have autocomplete either.

## A note on your editor's type checker

If you use VS Code with Pylance, or another editor that type-checks as you go, you will see squiggles on a handful of lines in `week03.py` — including the `card.color = "green"` line in the template's own test block, which was written for you.

They are expected and they are not your bug. All of them come from the same place: a type checker can only reason about what it can prove, and in a few spots this assignment knows something the checker cannot.

- In `__eq__`, the parameter is annotated `object`, because that is what Python requires of `__eq__`. After `type(self) is type(other)` you know `other` is the same class you are, but the checker does not follow that reasoning, so it objects to `other._size` or `other._rank`.
- In the test block, `card.color = "green"` deliberately assigns to a read-only property to show that it raises. The checker is right that this cannot work — that is the point of the line.

The autograder runs a different checker, `ty`, which accepts all of these, so **none of them costs you marks**. Write the straightforward code; do not contort it to silence an editor. If a squiggle appears somewhere else, on a line you wrote, that one is worth reading.

---

## Problem 1: Dynamic Array Accessors

In class we built a `DynamicArray`: a fixed-size block of slots that doubles its capacity whenever it runs out of room. A Python list does this for you automatically; the point of building one yourself is to see what `append()` actually costs.

The template's class keeps three pieces of state:

- `_underlying` -- a plain Python list standing in for the raw array. It **always** holds exactly `_capacity` entries.
- `_capacity` -- the total number of slots, including the empty ones.
- `_size` -- how many of those slots actually hold a value.

Empty slots hold the **sentinel** value `_EMPTY`, which is `-1`. A sentinel is a stand-in meaning "nothing here yet". It only works as long as `-1` can never be real data, which is why this class stores non-negative integers only -- and why `_size`, not the contents, is the real source of truth about how much is stored.

`_size` does double duty, exactly as it did in class: it is both the count of stored values **and** the index of the next free slot.

Your task is to implement the seven accessor methods listed under **What to Submit** below.

### Example

```
da = DynamicArray()          # capacity 4, size 0
da.add(10001)
da.add(60626)
da.add(90210)

len(da)              -> 3
da.get_size()        -> 3
da.get_capacity()    -> 4
```

**Explanation:** three values stored in an array of four slots. The fourth slot still holds the sentinel, so size is 3 while capacity is 4. They are equal only when the array is exactly full.

### Example

```
da.get(0)     -> 10001
da.get(2)     -> 90210
da.get(-1)    -> -1
da.get(3)     -> -1
da.get(100)   -> -1
```

**Explanation:** only positions `0` through `_size - 1` hold values. Position `3` is a real slot but nothing has been stored there, so it is out of range just as `100` is. Position `-1` is out of range too -- note that a plain Python list would have happily returned the *last* element for `da._underlying[-1]`, so this case needs a bounds check you write yourself.

### Example

```
da.index_of(10001)   -> 0
da.index_of(99999)   -> -1
da.contains(60626)   -> True
da.contains(99999)   -> False
```

**Explanation:** `index_of` is the linear search we refactored in class; `contains` answers the same question with a `bool`.

### Example

```
da                   # [10001, 60626, 90210, 11111, 22222], capacity 8
da.remove(90210)     -> True
da                   # [10001, 60626, 11111, 22222]
da.get_size()        -> 4
da.get_capacity()    -> 8
da.remove(90210)     -> False
da.index_of(22222)   -> 3
```

**Explanation:** removing from the middle leaves a hole, so `11111` and `22222` each shift one slot to the left and `22222` ends up at index 3. The slot freed at the end goes back to the sentinel and `_size` drops by one. **Capacity does not change** -- the array never shrinks. Removing a value that is not there changes nothing and reports `False`.

## Constraints

- Every value stored is a **non-negative integer**. The template's `add` raises `ValueError` for anything negative, because a negative value could not be told apart from the `-1` sentinel. You do not need to re-check this in your own methods.
- `get` and `index_of` must **not** raise for an out-of-range index or a missing value -- they report the sentinel instead. `remove` reports `False`. Returning the sentinel *is* this class's error signal.
- `self._EMPTY` and a bare `-1` are the same value, and **either spelling is accepted** -- `-1` is never a magic value under the Pact. Use whichever reads better to you; this document uses both.
- `index_of` must search the **filled slots only** (positions `0` through `_size - 1`). The sentinel slots all hold `-1`, so searching them would let `index_of` report a slot that holds nothing.
- `contains` must be written **in terms of `index_of`**, not as a second search loop. This is the duplicated-logic refactor we did in class, and it is checked directly on your submitted code.
- `remove` must not change `_capacity`, and must leave the vacated slot holding `_EMPTY`.
- `week03.py` may **not** contain any `import` statements at all.
- The [Programmer's Pact](../../programmers-pact.md) applies to everything you write: one `return` per value-returning method, no `break`/`continue`, comments, no magic values. Note that `0`, `+1`, and `-1` are never magic values and may be written directly.

## What to Submit

Implement these seven methods in `week03.py`, keeping the signatures exactly as the template gives them:

| Method | Returns |
| --- | --- |
| `__len__(self) -> int` | the number of values stored |
| `get_size(self) -> int` | the number of values stored |
| `get_capacity(self) -> int` | the total number of slots, sentinels included |
| `get(self, index: int) -> int` | the value at `index`, or `-1` if out of range |
| `index_of(self, value: int) -> int` | the position of the first occurrence of `value`, or `-1` |
| `contains(self, value: int) -> bool` | whether `value` is stored |
| `remove(self, value: int) -> bool` | removes the first occurrence; whether anything was removed |

---

## Problem 2: Dynamic Array Magic Methods

This problem extends the same `DynamicArray` class from Problem 1, in the same file. Finish Problem 1 first -- three of the four methods here are much easier to write once `add` and the accessors are working.

*Magic methods* (or *dunder* methods, for the double underscores around their names) are how a class teaches Python's built-in syntax to work on it. You rarely call them yourself: Python calls `__len__` when you write `len(da)`, `__eq__` when you write `a == b`, and `__add__` when you write `a + b`. Define them and your object behaves like a built-in type; leave them out and Python falls back to defaults that are technically correct and useless, like `<__main__.DynamicArray object at 0x104e2f0>`.

You will write four:

- `from_values(values)` -- an **alternative constructor**. Not a magic method itself, but `__repr__` needs it. It is a `@classmethod`, so it receives the class as `cls` instead of an instance as `self`, exactly like `Circle.from_diameter` from class.
- `__eq__(other)` -- what `a == b` means for two arrays.
- `__repr__()` -- what a programmer sees at the shell.
- `__add__(other)` -- what `a + b` builds.

### `__str__` and `__repr__` are for different readers

`__str__` (already written for you) is for a human reading output: `[10001, 60626]`. `__repr__` is for a programmer, and the goal is a string that could **rebuild the object**. That is why `from_values` has to exist first -- `__init__` only takes a capacity, so there is no way to write a rebuilding string in terms of it.

`__repr__` is also what Python shows for an object sitting inside a list, which is why a list of arrays prints readably once you have written it.

### Example

```
a = DynamicArray.from_values([10001, 60626, 90210])

a                    # at the shell, shows:
                     # DynamicArray.from_values([10001, 60626, 90210])
print(a)             # [10001, 60626, 90210]
repr(DynamicArray()) # 'DynamicArray.from_values([])'
```

**Explanation:** `__repr__` produces the rebuilding string, `__str__` the human-readable one. An empty array still reports a valid rebuilding string.

### Example

```
a = DynamicArray.from_values([10001, 60626, 90210])
b = DynamicArray.from_values([10001, 60626, 90210])

a == b     -> True
a is b     -> False
```

**Explanation:** two separate objects holding the same values are equal. `is` asks whether they are the *same object*, which they are not -- that distinction is the reason `__eq__` has to exist at all. Without it, Python compares identity and `a == b` would be `False`.

### Example

```
wide = DynamicArray(64)        # capacity 64
wide.add(10001); wide.add(60626); wide.add(90210)

a == wide             -> True
a.get_capacity()      -> 4
wide.get_capacity()   -> 64
```

**Explanation:** **capacity is not part of equality.** Two arrays holding the same values in the same order are equal no matter how much unused room each one has. Capacity is an implementation detail; the values are the identity.

### Example

```
a == DynamicArray.from_values([10001, 60626])          -> False   (different length)
a == DynamicArray.from_values([90210, 60626, 10001])   -> False   (order matters)
a == "not an array"                                     -> False
a == None                                               -> False
```

**Explanation:** comparing against something that is not a `DynamicArray` returns `False` rather than raising.

### Example

```
c = a + DynamicArray.from_values([11111, 22222])

c          # [10001, 60626, 90210, 11111, 22222]
len(c)     -> 5
a          # [10001, 60626, 90210]   <- unchanged
len(a)     -> 3                      <- unchanged
```

**Explanation:** `a + b` builds and returns a **new** array. Neither operand is modified -- this is the same rule arithmetic follows, where `x + y` does not change `x`.

## Constraints

- `from_values` must build its result with `cls(...)`, not with `DynamicArray(...)`. Called on a subclass, it has to produce an instance of that subclass -- this matters in a later problem.
- `__eq__` must compare `type(self)` with `type(other)` rather than using `isinstance`. An instance of a subclass must **not** be equal to a plain `DynamicArray` holding the same values.
- `__eq__` must return `False` for a non-array argument (`None`, a string, a number), never raise.
- `__eq__` must ignore capacity entirely.
- `__repr__` must build the class name with `type(self).__name__` rather than hardcoding `"DynamicArray"`, so a subclass reports its own name. Its output must be a string that, pasted back into Python, rebuilds an equal array.
- `__add__` must not modify either operand, and must not return one of them. It returns a plain `DynamicArray`.
- The [Programmer's Pact](../../programmers-pact.md) still applies: one `return` per value-returning method, no `break`/`continue`, comments, no magic values.
- A note, not a requirement: defining `__eq__` makes the class **unhashable**, so a `DynamicArray` can no longer be put in a `set` or used as a `dict` key. Python does this deliberately -- objects that compare equal would need equal hashes, and it will not guess one for you.

## What to Submit

Implement these four methods in the same `week03.py`, keeping the signatures exactly as the template gives them:

| Method | Returns |
| --- | --- |
| `from_values(cls, values: list[int]) -> DynamicArray` | a new array holding `values`, in order |
| `__eq__(self, other: object) -> bool` | whether `other` is the same type and holds the same values in the same order |
| `__repr__(self) -> str` | a string that rebuilds this array |
| `__add__(self, other: DynamicArray) -> DynamicArray` | a new array: this one's values followed by `other`'s |

---

## Problem 3: Cards, Properties and Decorators

This problem is independent of Problems 1 and 2 — a different class, in the same file. It picks up the `Card` class from the Week 3-2 handout and the four tools we met alongside it: `@staticmethod`, `@property`, a property setter, and a decorator you write yourself.

The template gives you `Card.__init__`, `label()`, `__eq__`, and two class constants, `_RANKS` (the 13 legal ranks) and `_RED_SUITS`.

### Refresher: what the `@` actually does

All four tools here come straight from the **Week 3-2 handout** — Activity 1, whose written solutions work through each one. Reread it if any of this is hazy; the short version is:

- A decorator takes the object defined below it and returns the object that the name should point at instead. So `@announce` above `def describe(...)` means `describe = announce(describe)`.
- `@property` is that same rule applied inside a class body, which is why `type(Card.rank)` is `property` while `type(Card.label)` is `function`.
- `@rank.setter` goes above a **second method also named `rank`**: the name you decorate is the name that gets rebound, so naming it `set_rank` rebinds nothing.
- A property with a getter and no setter is **read-only** — assigning to it raises `AttributeError`.


### Example

```
card = Card("7", "hearts")

card.label()            -> '7 of hearts'
card.rank               -> '7'          # runs the getter, no parentheses
card.color              -> 'red'
Card("K", "spades").color  -> 'black'
```

**Explanation:** `label()` is a method and is called with parentheses. `rank` and `color` are properties, read like attributes.

### Example

```
Card.is_valid_rank("Q")    -> True
Card.is_valid_rank("1")    -> False
card.is_valid_rank("10")   -> True
```

**Explanation:** `is_valid_rank` is a `@staticmethod` — the question is about a rank, not about a particular card, so it needs neither `self` nor `cls`. It can be called on the class or on an instance.

### Example

```
card.rank = "K"
card.rank        -> 'K'
card.label()     -> 'K of hearts'

card.rank = "1"  -> ValueError: '1' is not one of the 13 legal ranks
card.rank        -> '7'      # unchanged, the assignment was rejected
```

**Explanation:** assignment runs the setter, which validates first. A plain attribute would have accepted `"1"` without a word.

### Example

```
card.color = "green"   -> AttributeError
```

**Explanation:** `color` has a getter and no setter, so it is read-only.

### Example

```
repr(card)     -> "Card('7', 'hearts')"
[card, card]   -> [Card('7', 'hearts'), Card('7', 'hearts')]
```

**Explanation:** note the **quotes** around `7` and `hearts`. Typing `Card('7', 'hearts')` back into Python rebuilds an equal card; `Card(7, hearts)` — what plain string concatenation would give you — rebuilds nothing. A list prints by calling `repr` on each element, never `str`.

### Example

```
@announce
def describe(rank: str, suit: str) -> str:
    return f"{rank} of {suit}"

print(describe("7", "hearts"))
# calling describe
# 7 of hearts

describe.__name__    -> 'wrapper'
```

**Explanation:** two lines appear. The first comes from inside the wrapper, before the wrapped function runs; the second is the value `print` received. The name `describe` now points at `wrapper`.

## Constraints

- `is_valid_rank` must be a `@staticmethod` and must not take `self` or `cls`.
- `rank`'s getter must be decorated `@property`, and its setter `@rank.setter`. Both methods are named `rank`.
- The setter must raise `ValueError` for any rank not in `_RANKS`, and must leave the card unchanged when it does. Reuse `is_valid_rank` rather than repeating the list of ranks.
- `color` must be computed from the suit each time, not stored in `__init__`, and must stay **read-only** — no setter.
- `__repr__` must produce a string that rebuilds an equal card, quotes included. Use an f-string with `!r`.
- `announce` must return a new function that prints `calling <name of the wrapped function>` and then returns whatever the wrapped function returns. Name the inner function `wrapper`. It must work for a wrapped function taking any number of positional arguments -- `describe` takes two, but your decorator should not assume that. The tests only ever call decorated functions positionally.
- **You must also apply it.** Writing `announce` is only half of a decorator; a decorator is applied. At the bottom of `week03.py`, inside the `if __name__ == "__main__":` block, a `TODO` comment marks a `describe` function that is deliberately left undecorated. Add the line `@announce` directly above its `def`. Until you do, `describe` is an ordinary function and the two prints after it will not match their expected output.
- **`announce` is the one function in this assignment exempt from the type-annotation requirement**, because annotating it needs `Callable`, and this assignment allows no imports. Leave it unannotated, as in the handout.
- Membership tests with `in` **are allowed here** (`rank in Card._RANKS`). The "no `in` for searching" rule from the Week 2-1 refactoring exercise is about writing a linear search by hand, which is what Problem 1's `index_of` is for; checking membership of a fixed 13-element constant is not that exercise.
- The [Programmer's Pact](../../programmers-pact.md) still applies: one `return` per value-returning method, no `break`/`continue`, comments, no magic values.

## What to Submit

Implement these in the same `week03.py`, keeping the signatures exactly as the template gives them:

| Name | Kind | Returns |
| --- | --- | --- |
| `is_valid_rank(rank: str) -> bool` | `@staticmethod` | whether `rank` is one of the 13 legal ranks |
| `rank` | `@property` getter | this card's rank |
| `rank` | `@rank.setter` | nothing; raises `ValueError` on an illegal rank |
| `color` | `@property`, read-only | `"red"` or `"black"`, computed from the suit |
| `__repr__(self) -> str` | method | a string that rebuilds this card, quotes included |
| `announce(func)` | module-level function | a wrapper that announces the call, then delegates |
| the `@announce` line | one added line in `__main__` | applies your decorator to `describe` |

---

## Problem 4: Bounded Arrays, Inheritance and Name Mangling

Back to the array. A `BoundedArray` **is a** `DynamicArray` that has taken on one extra promise: it will never hold more than a fixed number of values, however much room the underlying array happens to have.

Inheritance is how you say that in code. `class BoundedArray(DynamicArray)` gives the new class everything the old one has — `get`, `index_of`, `contains`, `remove`, `__len__`, `__repr__`, `from_values`, all of it — without copying a line. You then define only what you need to add or change.

### `super()` and overriding

`super().__init__()` calls the parent's initializer, which sets up `_underlying`, `_capacity` and `_size` exactly as `DynamicArray` does. Writing those three assignments again in the child would be duplicated logic of the worst kind: it works today and drifts the moment the parent changes.

`add` is the method you **override**. Your version checks the limit, then calls `super().add(value)` to do the actual storing. That is the whole pattern — the child adds a rule, the parent still does the work.

There is a payoff worth noticing. `from_values` was written back in Problem 2, and it builds its result with `cls()`, so `BoundedArray.from_values([...])` hands you a `BoundedArray`. It also calls `add` — *your* `add` — so the limit is enforced through a method written before `BoundedArray` existed. That is inheritance doing its job.

This is also why your `__init__` must stay callable with **no arguments**: `cls()` passes none.

### Two underscores, and what they really do

Store the limit in an attribute named with **two** leading underscores: `self.__limit`.

Inside a class body, Python rewrites such a name at compile time to `_ClassName__name`, using the class the line is *written in*, not the class of the object at runtime. So `self.__limit` inside `BoundedArray` becomes `self._BoundedArray__limit`, and `vars(b)` shows it under that rewritten key, sitting in plain sight next to `_capacity`, `_size` and `_underlying`.

This is **not** privacy. Nothing is hidden — the attribute is merely filed under a different name, and anyone who spells that name out can read it. What mangling actually buys is protection from a *subclass* accidentally reusing the same attribute name for something unrelated. One underscore says "internal, use at your own risk" to a caller; two says "this is mine, and a subclass writing `__limit` gets its own".

### Example

```
b = BoundedArray()
type(b).__name__    -> 'BoundedArray'
b.get_limit()       -> 6

for value in range(6):
    b.add(value)

len(b)              -> 6
b.get_capacity()    -> 8
b.add(99)           -> IndexError: array is full
```

**Explanation:** the parent would have resized and carried on; the override refuses. Note that the two numbers differ: capacity reached **8** by doubling from 4, while the limit is **6**. Capacity is how many slots exist; the limit is how many you are allowed to fill. Room in the array is not permission to use it.

### Example

```
pair = BoundedArray.from_values([1, 2])

type(pair).__name__   -> 'BoundedArray'
repr(pair)            -> 'BoundedArray.from_values([1, 2])'

BoundedArray.from_values([0, 1, 2, 3, 4, 5, 6])
    -> IndexError: array is full
```

**Explanation:** `from_values` is inherited from Problem 2 and was never told about `BoundedArray`. It returns the right class because it builds with `cls()`, and it enforces the limit because it calls `add`, which is now yours. `__repr__` reports the right name for the same reason — it uses `type(self).__name__`.

### Example

```
pair == DynamicArray.from_values([1, 2])   -> False
pair == BoundedArray.from_values([1, 2])   -> True
```

**Explanation:** the `__eq__` you wrote in Problem 2 compares `type(self)` with `type(other)`, so a `BoundedArray` is never equal to a plain `DynamicArray`, even holding identical values. They make different promises about what happens next.

### Example

```
sorted(vars(b))
-> ['_BoundedArray__limit', '_capacity', '_size', '_underlying']

b.__limit              -> AttributeError
b._BoundedArray__limit -> 6
```

**Explanation:** the name was rewritten, not hidden. `b.__limit` fails from outside the class because no attribute is stored under that spelling; `b._BoundedArray__limit` works because that is where it actually lives.

## Constraints

- `BoundedArray` must inherit from `DynamicArray` — `class BoundedArray(DynamicArray):`.
- `__init__` must call `super().__init__()` rather than setting up `_underlying`, `_capacity` or `_size` itself.
- `__init__` must be callable with **no arguments**, so `limit` needs a default. Inherited code builds instances with `cls()`, and that has to keep working.
- The limit must be stored in an attribute with **two** leading underscores, so that `vars()` shows it under its mangled name.
- `add` must override the parent's, raise `IndexError("array is full")` when the array already holds its limit, and otherwise delegate to `super().add(value)` rather than writing to `_underlying` directly.
- Do **not** override anything else. `get`, `index_of`, `contains`, `remove`, `__len__`, `__repr__` and `__eq__` are inherited and keep working exactly as they did.
- `from_values` is inherited too, but it does **not** behave identically, and that is correct. Because it calls `add`, it now goes through *your* `add` and enforces your limit: `BoundedArray.from_values([0, 1, 2, 3, 4, 5, 6])` raises `IndexError`. Inherited code calling overridden code is the whole point of the exercise, not a bug to design around.
- `__add__` is the exception in the other direction, and this is also deliberate. It was written in Problem 2 to return a **plain** `DynamicArray`, so adding two bounded arrays gives you an ordinary unbounded one that may well be longer than either limit. A bound is a promise a particular object makes about itself, not a property that survives being concatenated into something new.
- The [Programmer's Pact](../../programmers-pact.md) still applies: one `return` per value-returning method, no `break`/`continue`, comments, no magic values. Note that `6` is a magic value — the template names it for you as `_DEFAULT_LIMIT`.

## What to Submit

Implement these three in the same `week03.py`, keeping the signatures exactly as the template gives them:

| Method | Behavior |
| --- | --- |
| `__init__(self, limit: int = _DEFAULT_LIMIT) -> None` | calls `super().__init__()`, then stores `limit` under a doubly-underscored name |
| `get_limit(self) -> int` | returns the greatest number of values this array will hold |
| `add(self, value: int) -> None` | raises `IndexError("array is full")` at the limit, otherwise delegates to `super().add(value)` |

---

## Submitting on Gradescope

- Download the template [`week03.py`](./week03.py), fill in the methods marked with `NotImplementedError`, and submit **that file**, named exactly `week03.py`, to the **Week03** assignment on Gradescope. Don't submit a `.zip`, a notebook, or a renamed file -- the autograder expects that exact filename.
- Don't submit this notebook. The notebook is the instructions; `week03.py` is the answer.
- Run `uv run week03.py` locally before you submit. The template's `if __name__ == "__main__":` block prints each result next to the value it should have -- every line should match.
- You may resubmit as many times as you like before the deadline; only your last submission counts. Use this to iterate -- don't wait until you think you're completely done to submit once.
- Before submitting, read your code against the [Programmer's Pact](../../programmers-pact.md) and this problem's constraints, the same way the autograder will.
- Feedback is staged in three tiers:
    - **Visible immediately** on every submission: the basic examples from this notebook, plus the Pact/constraint checks (imports, comments, return discipline, etc.) -- these tell you right away if something structural is wrong.
    - **Hidden until the deadline**: a broader sweep of edge cases and invalid inputs. You won't see these results until grading closes, but they count toward your score the whole time.
    - **Released after grades are published**: a stress case with more detailed feedback, including a pointer to where in your code a failure happened -- meant to help you understand a mistake after the fact, not to help you pass the assignment.
- If the autograder's output doesn't make sense to you, or you think it's wrong, ask on the course forum or come to office hours rather than guessing repeatedly against the hidden tests.